## 1. Setup — Connessione a Google Drive ed estrazione dei dati

  Monta Google Drive sull'ambiente Colab e decomprime i due archivi ZIP nella directory locale `/content/tesi`:
  - `subset_manifest.zip` → il CSV con i metadati delle 29.250 immagini del subset
  - `output_subset.zip` → le immagini stesse, organizzate per cluster e categoria

In [1]:
from google.colab import drive
import zipfile
import pathlib

# 1. collego Google Drive
drive.mount('/content/drive')

# 2. definiamo i percorsi vari
# La cartella sorgente su Drive dove sono i due zip
BASE_PATH = pathlib.Path("/content/drive/MyDrive/tesi")

# La cartella di destinazione locale su Colab
DEST = pathlib.Path("/content/tesi")
DEST.mkdir(parents=True, exist_ok=True)

# 3. Elenco dei file da estrarre
zips_to_extract = ["subset_manifest.zip", "output_subset.zip"]

# 4. Ciclo di estrazione
for zip_name in zips_to_extract:
    zip_path = BASE_PATH / zip_name

    if zip_path.exists():
        print(f"Estrazione di {zip_name} in corso...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DEST)
        print(f"Completato: {zip_name}")
    else:
        print(f"Errore: Il file {zip_name} non è stato trovato in {BASE_PATH}")

print("\nTutti i file sono stati estratti in:", DEST)

Mounted at /content/drive
Estrazione di subset_manifest.zip in corso...
Completato: subset_manifest.zip
Estrazione di output_subset.zip in corso...
Completato: output_subset.zip

Tutti i file sono stati estratti in: /content/tesi


## 2. Installazione di CLIP e caricamento del modello

  Installa la libreria OpenAI CLIP direttamente da GitHub, importa le dipendenze e carica il modello **ViT-L/14** sulla GPU (T4 su Colab).

  ViT-L/14 produce embedding di dimensione **768**
  Il modello viene messo in
  modalità `eval()` per disabilitare dropout e batch norm in training mode.

In [2]:
!pip install git+https://github.com/openai/CLIP.git -q

import clip
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# uso la GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Caricamento del modello ViT-L/14
model, preprocess = clip.load("ViT-L/14", device=device)
model.eval()#modalità inferenza

print(f"Modello caricato: ViT-L/14")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
Device: cuda


100%|████████████████████████████████████████| 890M/890M [00:05<00:00, 175MiB/s]


Modello caricato: ViT-L/14


## 3. Caricamento del manifest e conversione dei percorsi

  Carica il CSV `subset_manifest.csv` (prodotto dalla Phase 5) che contiene i percorsi delle 29.250 immagini del subset.

  La funzione `to_colab_path` converte i percorsi Windows (con backslash, registrati nella Phase 5 su Windows) in percorsi POSIX validi su Colab: estrae la parte del path a
  partire da `output_subset/` e la ricostruisce sotto `/content/tesi/`. Al termine verifica che tutte le immagini siano effettivamente accessibili sul filesystem locale.

In [5]:
SUBSET_ROOT = Path("/content/tesi/output_subset")
MANIFEST = Path("/content/tesi/subset_manifest.csv")
BASE_TESI = SUBSET_ROOT.parent #avrei potuto tranquillamente ridefinire pathlib.Path("/content/drive/MyDrive/tesi"), ma è la stessa cosa

# Caricamento del manifest
df = pd.read_csv(MANIFEST)

def to_colab_path(win_path: str) -> Path:
    # 1. Puliamo il path Windows per isolare la parte da 'output_subset'
    p_str = win_path.replace('/', '\\') # Uniformiamo tutto a backslash
    parts = p_str.split('\\')

    try:
        idx = parts.index("output_subset")
        # Usando il backslash come carattere nel nome
        filename_with_backslashes = "\\".join(parts[idx:])
        return BASE_TESI / filename_with_backslashes
    except ValueError:
        return BASE_TESI / parts[-1]

# Applica e verifica
df["colab_path"] = df["image_path_dest"].apply(to_colab_path)
missing = df["colab_path"].apply(lambda p: not p.exists()).sum()

print(f"Esempio percorso cercato: {df['colab_path'].iloc[0]}")
print(f"Immagini trovate: {len(df) - missing}/{len(df)}")


if missing > 0:
    print("Nota: Controlla che i file siano stati estratti correttamente in:", SUBSET_ROOT)

Esempio percorso cercato: /content/tesi/output_subset\cluster_0\shopping_mall\indoor\00002254.jpg
Immagini trovate: 29250/29250


## 4. Estrazione degli embedding CLIP in batch

Esegue il forward pass del modello su tutte le 29.250 immagini in batch da 256 immagini ciascuno (115 batch totali).

Per ogni immagine:
1. **Preprocessing**: resize + crop a 224×224, normalizzazione canali RGB secondo i parametri CLIP
2. **Encoding**: `model.encode_image()` restituisce un vettore 768D



In [ ]:
BATCH_SIZE = 256
paths = df["colab_path"].tolist()
N = len(paths)

embeddings = np.zeros((N, 768), dtype=np.float32)
#ciclo le immagini a batch di 256 
for start in tqdm(range(0, N, BATCH_SIZE), desc="Batch"):
    end = min(start + BATCH_SIZE, N)
    batch_paths = paths[start:end]

    imgs = []
    for p in batch_paths:
        try:
            img = preprocess(Image.open(p).convert("RGB"))#resizo a 224 x 224 e converto le immagini in RGB 
            imgs.append(img)
        except Exception as e:
            print(f"[WARN] Skip {p}: {e}")
            imgs.append(torch.zeros(3, 224, 224))#se mi da errori creo un tensore vuoto al posto dell'immagine

    batch_tensor = torch.stack(imgs).to(device)#trasforma la lista di tensori in un unico tensore e lo sposta in GPU

    with torch.no_grad():#modalità inferenza
        feats = model.encode_image(batch_tensor)#estraggo gli embedding
        # Nessuna L2 norm: SAEDataset applica mean_center + target_norm=sqrt(768) internamente
        embeddings[start:end] = feats.cpu().float().numpy()#aggiungo gli embedding

print(f"Shape finale: {embeddings.shape}")
print(f"Norma media: {np.linalg.norm(embeddings, axis=1).mean():.2f}")#non sono normalizzati quindi non vengono tutti 1 

## 5. Salvataggio degli embedding su Drive

Salva i risultati in formato **raw memmap** (richiesto da `SAEDataset` della repo MSAE):

- Salvato con `np.memmap` (no header numpy), non con `np.save`
- Nome file segue la naming convention `{dataset}_{model}_{split}_image_{N}_{D}.npy` che `SAEDataset` usa per parsare N e D dal filename
- Split **train/val/test 70/20/10 stratificato per cluster** per garantire bilanciamento semantico
- Tre CSV manifest allineati (indice riga = indice embedding) per lookup categoria/cluster nelle fasi successive

In [ ]:
OUT = Path("/content/drive/MyDrive/tesi")
N, D = embeddings.shape  # 29250, 768
TRAIN_SIZE = 0.7
VAL_SIZE   = 0.2
# Split train/val/test 70/20/10 stratificato per cluster
rng = np.random.default_rng(42)
cluster_ids = df["cluster_id"].values

train_idx, val_idx, test_idx = [], [], []
for c in np.unique(cluster_ids):
    mask = np.where(cluster_ids == c)[0]
    rng.shuffle(mask)
    n = len(mask)
    split_train = int(n * TRAIN_SIZE)
    split_val   = int(n * (TRAIN_SIZE + VAL_SIZE))
    train_idx.extend(mask[:split_train].tolist())
    val_idx.extend(mask[split_train:split_val].tolist())
    test_idx.extend(mask[split_val:].tolist())

train_idx = np.array(sorted(train_idx))
val_idx   = np.array(sorted(val_idx))
test_idx  = np.array(sorted(test_idx))
N_train, N_val, N_test = len(train_idx), len(val_idx), len(test_idx)
print(f"Train: {N_train}  |  Val: {N_val}  |  Test: {N_test}")

# salva in formato raw binary, cioè i byte del dato scritti direttamente su disco, senza header, senza metadati, senza compressione
def save_memmap(path, data):
    mm = np.memmap(path, dtype=np.float32, mode="w+", shape=data.shape)
    mm[:] = data
    mm.flush()
    print(f"Salvato: {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")

save_memmap(
    OUT / f"places365_ViT-L~14_train_image_{N_train}_{D}.npy",
    embeddings[train_idx]
)
save_memmap(
    OUT / f"places365_ViT-L~14_val_image_{N_val}_{D}.npy",
    embeddings[val_idx]
)
save_memmap(
    OUT / f"places365_ViT-L~14_test_image_{N_test}_{D}.npy",
    embeddings[test_idx]
)

df_train = df.iloc[train_idx][["cluster_id", "category", "image_path_dest"]].copy()
df_train["embedding_idx"] = np.arange(N_train)
df_train.to_csv(OUT / "clip_image_embedding_manifest_train.csv", index=False)

df_val = df.iloc[val_idx][["cluster_id", "category", "image_path_dest"]].copy()
df_val["embedding_idx"] = np.arange(N_val)
df_val.to_csv(OUT / "clip_image_embedding_manifest_val.csv", index=False)

df_test = df.iloc[test_idx][["cluster_id", "category", "image_path_dest"]].copy()
df_test["embedding_idx"] = np.arange(N_test)
df_test.to_csv(OUT / "clip_image_embedding_manifest_test.csv", index=False)

print("Manifest salvati.")